# Fake Chat Models

The `fake_chat_models` module provides deterministic chat models for testing LangChain chains, agents, callbacks, streaming, tool calls, retries, and error handling without calling an external model provider.

# Import

The fake chat-model classes can be imported directly from the module.

**Syntax**

```python
from langchain_core.language_models.fake_chat_models import ( # Import fake chat-model classes
    FakeMessagesListChatModel, # Return predefined complete message objects
    FakeListChatModel, # Return predefined text responses sequentially
    FakeListChatModelError, # Represent a deliberate streaming failure
    FakeChatModel, # Always return the fixed text "fake response"
    GenericFakeChatModel, # Consume strings or AIMessage objects from an iterator
    ParrotFakeChatModel, # Return the final input message as the response
) # Finish importing the fake chat-model classes
```

# Common Public Methods

All fake chat models inherit the standard chat-model interface from `BaseChatModel`.

Inputs may be plain strings, `PromptValue` objects, or sequences containing LangChain messages or message-like values.

## Methods

1. `invoke`:= Performs one synchronous model invocation.

   The exact returned message type depends on the selected fake chat model.

   **Syntax**

   ```python
   invoke(
       self, # Fake chat-model instance
       input: LanguageModelInput, # String, prompt value, or message-like sequence
       config: RunnableConfig | None = None, # Runtime invocation configuration
       *,
       stop: list[str] | None = None, # Stop substrings
       **kwargs: Any # Additional invocation arguments
   ) -> BaseMessage
   ```

2. `ainvoke`:= Performs one asynchronous model invocation.

   This is the asynchronous equivalent of `invoke()`.

   **Syntax**

   ```python
   async ainvoke(
       self, # Fake chat-model instance
       input: LanguageModelInput, # String, prompt value, or message-like sequence
       config: RunnableConfig | None = None, # Runtime invocation configuration
       *,
       stop: list[str] | None = None, # Stop substrings
       **kwargs: Any # Additional invocation arguments
   ) -> BaseMessage
   ```

3. `stream`:= Streams one response synchronously as message chunks.

   Chunk boundaries depend on the selected fake model. Some fake models provide custom streaming behaviour, while others return the complete response as a single stream result.

   **Syntax**

   ```python
   stream(
       self, # Fake chat-model instance
       input: LanguageModelInput, # String, prompt value, or message-like sequence
       config: RunnableConfig | None = None, # Runtime invocation configuration
       *,
       stop: list[str] | None = None, # Stop substrings
       **kwargs: Any # Additional streaming arguments
   ) -> Iterator[BaseMessageChunk]
   ```

4. `astream`:= Streams one response asynchronously as message chunks.

   This is the asynchronous equivalent of `stream()`.

   **Syntax**

   ```python
   async astream(
       self, # Fake chat-model instance
       input: LanguageModelInput, # String, prompt value, or message-like sequence
       config: RunnableConfig | None = None, # Runtime invocation configuration
       *,
       stop: list[str] | None = None, # Stop substrings
       **kwargs: Any # Additional streaming arguments
   ) -> AsyncIterator[BaseMessageChunk]
   ```

5. `batch`:= Processes multiple inputs synchronously and returns one message for each input.

   The `config` parameter may contain one shared configuration for every input or one separate configuration for each input.

   When supported by the concrete fake model, `return_exceptions=True` returns failures inside the result list instead of raising them immediately.

   **Syntax**

   ```python
   batch(
       self, # Fake chat-model instance
       inputs: list[LanguageModelInput], # Model inputs to process
       config: RunnableConfig | list[RunnableConfig] | None = None, # Shared or per-input configuration
       *,
       return_exceptions: bool = False, # Return exceptions instead of raising them
       **kwargs: Any # Additional batch arguments
   ) -> list[BaseMessage]
   ```

6. `abatch`:= Processes multiple inputs asynchronously and returns one message for each input.

   This is the asynchronous equivalent of `batch()`.

   **Syntax**

   ```python
   async abatch(
       self, # Fake chat-model instance
       inputs: list[LanguageModelInput], # Model inputs to process
       config: RunnableConfig | list[RunnableConfig] | None = None, # Shared or per-input configuration
       *,
       return_exceptions: bool = False, # Return exceptions instead of raising them
       **kwargs: Any # Additional batch arguments
   ) -> list[BaseMessage]
   ```


In [ ]:
from langchain_core.language_models.fake_chat_models import FakeListChatModel # Import the fake chat model

model = FakeListChatModel( # Create a deterministic fake model
    responses=["First response", "Second response", "Third response"] # Define responses returned sequentially
) # Finish creating the model

response = model.invoke("Hello") # Perform one synchronous invocation
print(response.content) # Display the first configured response

async_response = await model.ainvoke("Hello again") # Perform one asynchronous invocation in Jupyter
print(async_response.content) # Display the second configured response

for chunk in model.stream("Stream this response"): # Stream the third response character by character
    print(chunk.content, end="") # Display each streamed character without a new line

print() # Move to the next output line

async for chunk in model.astream("Stream asynchronously"): # Stream the next response asynchronously
    print(chunk.content, end="") # Display each asynchronous character chunk

print() # Move to the next output line

batch_results = model.batch(["Input 1", "Input 2"]) # Process multiple inputs synchronously
print([message.content for message in batch_results]) # Display each batch response

async_batch_results = await model.abatch(["Input 3", "Input 4"]) # Process multiple inputs asynchronously
print([message.content for message in async_batch_results]) # Display each asynchronous batch response

## Classes

# FakeMessagesListChatModel: `BaseChatModel`

`FakeMessagesListChatModel` returns predefined complete LangChain message objects in sequence. After returning the final response, it cycles back to the first response.

It is useful when tests require control over complete message objects, including tool calls, identifiers, additional keyword arguments, response metadata, and usage metadata.

**Syntax**

```python
FakeMessagesListChatModel(
    responses: list[BaseMessage], # Complete messages returned sequentially
    sleep: float | None = None, # Optional delay before returning each response
    cache: BaseCache | bool | None = None, # Model-response cache configuration
    callbacks: Callbacks = None, # Callback handlers attached to model runs
    tags: list[str] | None = None, # Tags attached to model traces
    metadata: dict[str, Any] | None = None, # Metadata attached to model traces
    rate_limiter: BaseRateLimiter | None = None # Optional execution rate limiter
)
```

## Fields

1. `responses`:`list[BaseMessage]`:= Stores the complete messages returned by the model in their listed order.

   Common response types include `AIMessage` objects containing tool calls, additional keyword arguments, response metadata, or usage metadata. After the final response is returned, the model starts again from the first response.

2. `sleep`:`float | None`:= Stores an optional artificial delay in seconds before each response. Its default value is `None`.

3. `cache`:`BaseCache | bool | None`:= Controls LangChain model-response caching. Its default value is `None`.

4. `callbacks`:`Callbacks`:= Stores callback handlers attached to model executions. Its default value is `None`.

5. `tags`:`list[str] | None`:= Stores tags attached to traces generated by the model. Its default value is `None`.

6. `metadata`:`dict[str, Any] | None`:= Stores metadata attached to model traces. Its default value is `None`.

7. `rate_limiter`:`BaseRateLimiter | None`:= Stores an optional rate limiter applied before model execution. Its default value is `None`.

## Methods

1. `invoke`:= Returns the next complete message from `responses`.

   The configured message object is preserved, so fields such as `tool_calls`, `id`, `additional_kwargs`, `response_metadata`, and `usage_metadata` remain available.

   **Syntax**

   ```python
   invoke(
       self, # Fake message-list chat-model instance
       input: LanguageModelInput, # Input supplied to the model
       config: RunnableConfig | None = None, # Runtime invocation configuration
       *,
       stop: list[str] | None = None, # Stop substrings
       **kwargs: Any # Additional invocation arguments
   ) -> BaseMessage
   ```

2. `ainvoke`:= Asynchronously returns the next configured complete message.

   **Syntax**

   ```python
   async ainvoke(
       self, # Fake message-list chat-model instance
       input: LanguageModelInput, # Input supplied to the model
       config: RunnableConfig | None = None, # Runtime invocation configuration
       *,
       stop: list[str] | None = None, # Stop substrings
       **kwargs: Any # Additional invocation arguments
   ) -> BaseMessage
   ```

3. `batch`:= Processes multiple inputs and returns one configured response for each input.

   The response position continues advancing across batch and individual invocation calls.

   **Syntax**

   ```python
   batch(
       self, # Fake message-list chat-model instance
       inputs: list[LanguageModelInput], # Inputs to process
       config: RunnableConfig | list[RunnableConfig] | None = None, # Shared or per-input configuration
       *,
       return_exceptions: bool = False, # Return exceptions instead of raising them
       **kwargs: Any # Additional batch arguments
   ) -> list[BaseMessage]
   ```

4. `stream`:= Provides the standard synchronous chat-model streaming interface.

   This class does not divide the configured response into character-by-character or word-by-word chunks.

   **Syntax**

   ```python
   stream(
       self, # Fake message-list chat-model instance
       input: LanguageModelInput, # Input supplied to the model
       config: RunnableConfig | None = None, # Runtime invocation configuration
       *,
       stop: list[str] | None = None, # Stop substrings
       **kwargs: Any # Additional streaming arguments
   ) -> Iterator[BaseMessageChunk]
   ```


In [ ]:
from langchain_core.language_models.fake_chat_models import FakeMessagesListChatModel # Import the fake message-list model
from langchain_core.messages import AIMessage # Import the AI message class

responses = [ # Define complete messages returned by the model
    AIMessage( # Create the first predefined message
        content="Search completed.", # Store the message text
        id="message-1", # Store a message identifier
        tool_calls=[ # Store a tool call in the message
            { # Define the tool call
                "name": "search", # Specify the tool name
                "args": {"query": "LangChain"}, # Specify the tool arguments
                "id": "tool-1", # Store the tool-call identifier
                "type": "tool_call" # Specify the tool-call discriminator
            } # Finish defining the tool call
        ], # Finish the tool-call list
        response_metadata={"model": "fake-model"} # Store response metadata
    ), # Finish creating the first message
    AIMessage( # Create the second predefined message
        content="No tool call required.", # Store the second message text
        id="message-2" # Store the second message identifier
    ) # Finish creating the second message
] # Finish defining the responses

model = FakeMessagesListChatModel( # Create the fake chat model
    responses=responses, # Provide the predefined complete messages
    sleep=None, # Disable artificial delay
    cache=False, # Disable model-response caching
    tags=["test"], # Attach a tracing tag
    metadata={"purpose": "demo"} # Attach tracing metadata
) # Finish creating the model

first_response = model.invoke("Search for LangChain") # Return the first configured message
print(first_response.content) # Display the first message content
print(first_response.tool_calls) # Display the preserved tool call
print(first_response.response_metadata) # Display the preserved response metadata

second_response = await model.ainvoke("Continue") # Return the second message asynchronously in Jupyter
print(second_response.content) # Display the second message content

cycled_response = model.invoke("Start again") # Cycle back to the first configured message
print(cycled_response.id) # Display the first message identifier again

batch_results = model.batch(["Input 1", "Input 2"]) # Return one configured message per input
print([message.content for message in batch_results]) # Display the batch response contents

for chunk in model.stream("Stream a message"): # Stream the next complete response
    print(chunk.content) # Display the returned message-chunk content

# FakeListChatModel: `BaseChatModel`

`FakeListChatModel` returns predefined text responses sequentially. It supports character-by-character streaming and deliberate streaming failures.

After returning the final response, it cycles back to the first response.

**Syntax**

```python
FakeListChatModel(
    responses: list[str], # Text responses returned sequentially
    sleep: float | None = None, # Optional artificial delay in seconds
    error_on_chunk_number: int | None = None, # Stream-chunk index at which an error is raised
    cache: BaseCache | bool | None = None, # Model-response cache configuration
    callbacks: Callbacks = None, # Callback handlers attached to model runs
    tags: list[str] | None = None, # Tags attached to model traces
    metadata: dict[str, Any] | None = None, # Metadata attached to model traces
    rate_limiter: BaseRateLimiter | None = None # Optional execution rate limiter
)
```

## Fields

1. `responses`:`list[str]`:= Stores the text responses returned by the model in their listed order.

   Each selected string is converted into an `AIMessage` during `invoke()` or `ainvoke()`. After the final response is returned, the model starts again from the first response.

2. `sleep`:`float | None`:= Stores an optional artificial delay in seconds. Its default value is `None`.

   During `invoke()` and `ainvoke()`, the delay is applied once before returning the complete response. During `stream()` and `astream()`, the delay is applied before every character chunk.

3. `error_on_chunk_number`:`int | None`:= Stores the zero-based stream-chunk index at which `FakeListChatModelError` is raised. Its default value is `None`.

   An index of `0` raises the error before the first character is yielded. A value of `None` disables deliberate streaming failure.

4. `cache`:`BaseCache | bool | None`:= Controls LangChain model-response caching. Its default value is `None`.

5. `callbacks`:`Callbacks`:= Stores callback handlers attached to model executions. Its default value is `None`.

6. `tags`:`list[str] | None`:= Stores tags attached to model traces. Its default value is `None`.

7. `metadata`:`dict[str, Any] | None`:= Stores metadata attached to model traces. Its default value is `None`.

8. `rate_limiter`:`BaseRateLimiter | None`:= Stores an optional rate limiter applied before model execution. Its default value is `None`.

## Methods

1. `invoke`:= Returns the next configured text response as an `AIMessage`.

   The supplied input does not determine the response text.

   **Syntax**

   ```python
   invoke(
       self, # Fake list chat-model instance
       input: LanguageModelInput, # Input supplied to the model
       config: RunnableConfig | None = None, # Runtime invocation configuration
       *,
       stop: list[str] | None = None, # Stop substrings
       **kwargs: Any # Additional invocation arguments
   ) -> AIMessage
   ```

2. `ainvoke`:= Asynchronously returns the next configured text response as an `AIMessage`.

   **Syntax**

   ```python
   async ainvoke(
       self, # Fake list chat-model instance
       input: LanguageModelInput, # Input supplied to the model
       config: RunnableConfig | None = None, # Runtime invocation configuration
       *,
       stop: list[str] | None = None, # Stop substrings
       **kwargs: Any # Additional invocation arguments
   ) -> AIMessage
   ```

3. `stream`:= Streams the selected response synchronously, one character at a time.

   Every yielded value is an `AIMessageChunk`. The final character chunk is marked as the last chunk.

   When `error_on_chunk_number` matches the current character index, `FakeListChatModelError` is raised before that character is yielded.

   **Syntax**

   ```python
   stream(
       self, # Fake list chat-model instance
       input: LanguageModelInput, # Input supplied to the model
       config: RunnableConfig | None = None, # Runtime invocation configuration
       *,
       stop: list[str] | None = None, # Stop substrings
       **kwargs: Any # Additional streaming arguments
   ) -> Iterator[AIMessageChunk]
   ```

4. `astream`:= Asynchronously streams the selected response one character at a time.

   It uses the same `error_on_chunk_number` behaviour as `stream()`.

   **Syntax**

   ```python
   async astream(
       self, # Fake list chat-model instance
       input: LanguageModelInput, # Input supplied to the model
       config: RunnableConfig | None = None, # Runtime invocation configuration
       *,
       stop: list[str] | None = None, # Stop substrings
       **kwargs: Any # Additional streaming arguments
   ) -> AsyncIterator[AIMessageChunk]
   ```

5. `batch`:= Processes multiple inputs synchronously and returns one `AIMessage` for each input.

   Inputs are processed sequentially rather than concurrently so that predefined response ordering remains predictable.

   The first input receives the next response, the second input receives the following response, and so on.

   In the current implementation, `return_exceptions=True` does not convert failures into returned exception objects. Failures are allowed to propagate.

   **Syntax**

   ```python
   batch(
       self, # Fake list chat-model instance
       inputs: list[LanguageModelInput], # Inputs to process sequentially
       config: RunnableConfig | list[RunnableConfig] | None = None, # Shared or per-input configuration
       *,
       return_exceptions: bool = False, # Compatibility option that does not suppress failures
       **kwargs: Any # Additional batch arguments
   ) -> list[AIMessage]
   ```

6. `abatch`:= Asynchronously processes multiple inputs in explicit sequence and returns one `AIMessage` for each input.

   Sequential processing preserves the order of the predefined responses.

   **Syntax**

   ```python
   async abatch(
       self, # Fake list chat-model instance
       inputs: list[LanguageModelInput], # Inputs to process sequentially
       config: RunnableConfig | list[RunnableConfig] | None = None, # Shared or per-input configuration
       *,
       return_exceptions: bool = False, # Compatibility option that does not suppress failures
       **kwargs: Any # Additional batch arguments
   ) -> list[AIMessage]
   ```


In [ ]:
from langchain_core.language_models.fake_chat_models import FakeListChatModel, FakeListChatModelError # Import the fake model and its streaming error

model = FakeListChatModel( # Create a deterministic fake chat model
    responses=["First", "Second", "Third"], # Define responses returned sequentially
    sleep=None, # Disable artificial delays
    cache=False # Disable response caching
) # Finish creating the model

response = model.invoke("Any input") # Return the first configured response
print(response.content) # Display "First"

async_response = await model.ainvoke("Another input") # Return the second response asynchronously
print(async_response.content) # Display "Second"

for chunk in model.stream("Stream input"): # Stream the third response character by character
    print(chunk.content, end="") # Display each character without starting a new line
print() # Move to the next output line

cycled_response = model.invoke("Cycle input") # Cycle back to the first configured response
print(cycled_response.content) # Display "First"

batch_results = model.batch(["Input 1", "Input 2"]) # Process two inputs sequentially
print([message.content for message in batch_results]) # Display the next two responses

async_batch_results = await model.abatch(["Input 3", "Input 4"]) # Process two inputs asynchronously in sequence
print([message.content for message in async_batch_results]) # Display the ordered responses

error_model = FakeListChatModel( # Create a model configured to fail during streaming
    responses=["Hello"], # Define the response that would be streamed
    error_on_chunk_number=2 # Raise an error before yielding the third character
) # Finish creating the error-producing model

try: # Start handling the expected streaming error
    for chunk in error_model.stream("Test failure"): # Attempt to stream the response
        print(chunk.content, end="") # Display characters yielded before the failure
except FakeListChatModelError: # Catch the deliberate streaming failure
    print("\nStreaming failed on chunk index 2.") # Display an explanatory message

# FakeListChatModelError: `Exception`

`FakeListChatModelError` represents the deliberate streaming failure raised by `FakeListChatModel`.

It is raised only when `error_on_chunk_number` is configured and synchronous or asynchronous streaming reaches the selected zero-based chunk index.

**Syntax**

```python
class FakeListChatModelError(Exception): # Define the deliberate streaming error
    ... # No additional fields or methods are declared
```

## Behaviour

1. `error_on_chunk_number`:= Determines the character-chunk index at which `FakeListChatModelError` is raised by `FakeListChatModel`.

2. The error is raised before the character at the configured index is yielded.

3. The exception can be caught directly when testing retry behaviour, fallback models, callback error handling, partially completed streams, and agent or chain failure handling.


In [ ]:
from langchain_core.language_models.fake_chat_models import FakeListChatModel, FakeListChatModelError # Import the fake model and its deliberate streaming error

model = FakeListChatModel( # Create a fake chat model
    responses=["Hello"], # Define the response to stream
    error_on_chunk_number=2 # Raise an error before yielding character index 2
) # Finish creating the model

try: # Handle the expected streaming failure
    for chunk in model.stream("Test input"): # Stream the response character by character
        print(chunk.content, end="") # Display each character produced before the failure
except FakeListChatModelError: # Catch the deliberate streaming error
    print("\nFakeListChatModelError was raised.") # Display confirmation of the failure

# FakeChatModel: `BaseChatModel`

`FakeChatModel` is a deterministic chat model that returns the fixed text `"fake response"` for every invocation.

It does not require a predefined response list, and the supplied input does not affect the returned content.

**Syntax**

```python
FakeChatModel(
    cache: BaseCache | bool | None = None, # Model-response cache configuration
    callbacks: Callbacks = None, # Callback handlers attached to model runs
    tags: list[str] | None = None, # Tags attached to model traces
    metadata: dict[str, Any] | None = None, # Metadata attached to model traces
    rate_limiter: BaseRateLimiter | None = None # Optional execution rate limiter
)
```

## Fields

1. `cache`:`BaseCache | bool | None`:= Controls LangChain model-response caching. Its default value is `None`.

2. `callbacks`:`Callbacks`:= Stores callback handlers attached to model executions. Its default value is `None`.

3. `tags`:`list[str] | None`:= Stores tags attached to traces generated by the model. Its default value is `None`.

4. `metadata`:`dict[str, Any] | None`:= Stores metadata attached to model traces. Its default value is `None`.

5. `rate_limiter`:`BaseRateLimiter | None`:= Stores an optional rate limiter applied before model execution. Its default value is `None`.

## Methods

1. `invoke`:= Returns an `AIMessage` containing the fixed text `"fake response"`.

   The supplied input does not change the returned content.

   **Syntax**

   ```python
   invoke(
       self, # Fake chat-model instance
       input: LanguageModelInput, # Input supplied to the model
       config: RunnableConfig | None = None, # Runtime invocation configuration
       *,
       stop: list[str] | None = None, # Stop substrings
       **kwargs: Any # Additional invocation arguments
   ) -> AIMessage
   ```

2. `ainvoke`:= Asynchronously returns an `AIMessage` containing the fixed text `"fake response"`.

   **Syntax**

   ```python
   async ainvoke(
       self, # Fake chat-model instance
       input: LanguageModelInput, # Input supplied to the model
       config: RunnableConfig | None = None, # Runtime invocation configuration
       *,
       stop: list[str] | None = None, # Stop substrings
       **kwargs: Any # Additional invocation arguments
   ) -> AIMessage
   ```

3. `stream`:= Provides the standard inherited synchronous streaming behaviour.

   It does not support configurable character-by-character streaming like `FakeListChatModel`.

   **Syntax**

   ```python
   stream(
       self, # Fake chat-model instance
       input: LanguageModelInput, # Input supplied to the model
       config: RunnableConfig | None = None, # Runtime invocation configuration
       *,
       stop: list[str] | None = None, # Stop substrings
       **kwargs: Any # Additional streaming arguments
   ) -> Iterator[AIMessageChunk]
   ```

4. `batch`:= Processes multiple inputs and returns one fixed `"fake response"` message for each input.

   **Syntax**

   ```python
   batch(
       self, # Fake chat-model instance
       inputs: list[LanguageModelInput], # Inputs to process
       config: RunnableConfig | list[RunnableConfig] | None = None, # Shared or per-input configuration
       *,
       return_exceptions: bool = False, # Return exceptions instead of raising them
       **kwargs: Any # Additional batch arguments
   ) -> list[AIMessage]
   ```


In [ ]:
from langchain_core.language_models.fake_chat_models import FakeChatModel # Import the fixed-response fake chat model

model = FakeChatModel( # Create the fake chat-model instance
    cache=False, # Disable response caching
    tags=["demo"], # Attach a tracing tag
    metadata={"purpose": "testing"} # Attach tracing metadata
) # Finish creating the model

response = model.invoke("What is Python?") # Perform a synchronous invocation
print(response.content) # Display "fake response"

async_response = await model.ainvoke("What is LangChain?") # Perform an asynchronous invocation in Jupyter
print(async_response.content) # Display "fake response"

for chunk in model.stream("Stream a response"): # Stream the fixed response
    print(chunk.content, end="") # Display the returned chunk content

print() # Move to the next output line

batch_results = model.batch([ # Process multiple inputs
    "First input", # Provide the first input
    "Second input", # Provide the second input
    "Third input" # Provide the third input
]) # Finish the batch invocation

print([message.content for message in batch_results]) # Display one "fake response" for every input

# GenericFakeChatModel: `BaseChatModel`

`GenericFakeChatModel` consumes predefined strings or `AIMessage` objects from an iterator. Each response is consumed once and is not repeated automatically.

It is recommended for unit testing model, chain, and agent behaviour where responses must be supplied in a controlled order.

**Syntax**

```python
GenericFakeChatModel(
    messages: Iterator[AIMessage | str], # Iterator containing responses consumed sequentially
    cache: BaseCache | bool | None = None, # Model-response cache configuration
    callbacks: Callbacks = None, # Callback handlers attached to model runs
    tags: list[str] | None = None, # Tags attached to model traces
    metadata: dict[str, Any] | None = None, # Metadata attached to model traces
    rate_limiter: BaseRateLimiter | None = None # Optional execution rate limiter
)
```

## Fields

1. `messages`:`Iterator[AIMessage | str]`:= Stores the iterator containing responses consumed sequentially by the model.

   Each invocation consumes one value. A string is converted into an `AIMessage`, while an existing `AIMessage` preserves fields such as tool calls, identifiers, metadata, and `additional_kwargs`.

   An iterator must be supplied instead of an ordinary list. Responses do not cycle after the iterator is exhausted.

2. `cache`:`BaseCache | bool | None`:= Controls LangChain model-response caching. Its default value is `None`.

3. `callbacks`:`Callbacks`:= Stores callback handlers attached to model executions. Its default value is `None`.

4. `tags`:`list[str] | None`:= Stores tags attached to traces generated by the model. Its default value is `None`.

5. `metadata`:`dict[str, Any] | None`:= Stores metadata attached to model traces. Its default value is `None`.

6. `rate_limiter`:`BaseRateLimiter | None`:= Stores an optional rate limiter applied before model execution. Its default value is `None`.

## Methods

1. `invoke`:= Consumes and returns the next response from the `messages` iterator.

   A string response is converted into an `AIMessage`. An existing `AIMessage` retains its configured fields.

   The invocation input does not select or generate the response. Response order is controlled entirely by the iterator.

   `StopIteration` is raised when the iterator has been exhausted.

   **Syntax**

   ```python
   invoke(
       self, # Generic fake chat-model instance
       input: LanguageModelInput, # Input supplied to the model
       config: RunnableConfig | None = None, # Runtime invocation configuration
       *,
       stop: list[str] | None = None, # Stop substrings
       **kwargs: Any # Additional invocation arguments
   ) -> AIMessage
   ```

2. `ainvoke`:= Asynchronously consumes and returns the next response from the iterator.

   **Syntax**

   ```python
   async ainvoke(
       self, # Generic fake chat-model instance
       input: LanguageModelInput, # Input supplied to the model
       config: RunnableConfig | None = None, # Runtime invocation configuration
       *,
       stop: list[str] | None = None, # Stop substrings
       **kwargs: Any # Additional invocation arguments
   ) -> AIMessage
   ```

3. `stream`:= Consumes one response and streams its text content as whitespace-preserving chunks.

   Words and whitespace are emitted separately so the chunks can be recombined into the exact original text.

   The original `AIMessage` identifier is preserved in streamed chunks. Registered callbacks receive `on_llm_new_token` events for emitted text chunks.

   Values stored in `additional_kwargs` are also emitted through message chunks. Legacy `function_call` data is divided into smaller function-call argument chunks.

   A `ValueError` may be raised when the configured message contains an unsupported content form.

   **Syntax**

   ```python
   stream(
       self, # Generic fake chat-model instance
       input: LanguageModelInput, # Input supplied to the model
       config: RunnableConfig | None = None, # Runtime invocation configuration
       *,
       stop: list[str] | None = None, # Stop substrings
       **kwargs: Any # Additional streaming arguments
   ) -> Iterator[AIMessageChunk]
   ```

4. `astream`:= Asynchronously streams the next configured response through the standard chat-model streaming interface.

   **Syntax**

   ```python
   async astream(
       self, # Generic fake chat-model instance
       input: LanguageModelInput, # Input supplied to the model
       config: RunnableConfig | None = None, # Runtime invocation configuration
       *,
       stop: list[str] | None = None, # Stop substrings
       **kwargs: Any # Additional streaming arguments
   ) -> AsyncIterator[AIMessageChunk]
   ```

5. `batch`:= Processes multiple inputs and consumes one iterator response for each processed input.

   The iterator must contain enough responses for every model invocation.

   **Syntax**

   ```python
   batch(
       self, # Generic fake chat-model instance
       inputs: list[LanguageModelInput], # Inputs to process
       config: RunnableConfig | list[RunnableConfig] | None = None, # Shared or per-input configuration
       *,
       return_exceptions: bool = False, # Return exceptions instead of raising them
       **kwargs: Any # Additional batch arguments
   ) -> list[AIMessage]
   ```


In [ ]:
from langchain_core.language_models.fake_chat_models import GenericFakeChatModel # Import the iterator-based fake chat model
from langchain_core.messages import AIMessage # Import the AI message class

responses = iter([ # Create an iterator containing predefined responses
    "First plain-text response", # Convert this string into an AIMessage
    AIMessage( # Define a complete AIMessage response
        content="Second response with metadata", # Store the message text
        id="message-2", # Store the message identifier
        response_metadata={"model": "fake-model"} # Store response metadata
    ), # Finish defining the AIMessage
    "Stream this text exactly" # Use this response for streaming
]) # Finish creating the response iterator

model = GenericFakeChatModel( # Create the generic fake chat model
    messages=responses, # Supply the iterator consumed by model calls
    cache=False, # Disable response caching
    tags=["unit-test"], # Attach a tracing tag
    metadata={"purpose": "demonstration"} # Attach tracing metadata
) # Finish creating the model

first_response = model.invoke("First input") # Consume the first iterator value
print(first_response.content) # Display the converted string response

second_response = model.invoke("Second input") # Consume the predefined AIMessage
print(second_response.content) # Display the preserved message content
print(second_response.id) # Display the preserved message identifier
print(second_response.response_metadata) # Display the preserved response metadata

streamed_text = "" # Create a variable for rebuilding the streamed response

for chunk in model.stream("Third input"): # Consume and stream the third iterator value
    print(repr(chunk.content)) # Display each word or whitespace-preserving chunk
    streamed_text += chunk.content # Recombine the chunks into the original text

print(streamed_text) # Display the exact reconstructed response

try: # Test behaviour after the iterator is exhausted
    model.invoke("Fourth input") # Attempt to consume another response
except StopIteration: # Catch the exhausted-iterator error
    print("No configured responses remain.") # Display an exhaustion message

# ParrotFakeChatModel: `BaseChatModel`

`ParrotFakeChatModel` returns the final message from the supplied input instead of generating a separate AI response.

It does not accept predefined responses. When multiple messages are supplied, all earlier messages are ignored and only the final message is returned.

**Syntax**

```python
ParrotFakeChatModel(
    cache: BaseCache | bool | None = None, # Model-response cache configuration
    callbacks: Callbacks = None, # Callback handlers attached to model runs
    tags: list[str] | None = None, # Tags attached to model traces
    metadata: dict[str, Any] | None = None, # Metadata attached to model traces
    rate_limiter: BaseRateLimiter | None = None # Optional execution rate limiter
)
```

## Fields

1. `cache`:`BaseCache | bool | None`:= Controls LangChain model-response caching. Its default value is `None`.

2. `callbacks`:`Callbacks`:= Stores callback handlers attached to model executions. Its default value is `None`.

3. `tags`:`list[str] | None`:= Stores tags attached to traces generated by the model. Its default value is `None`.

4. `metadata`:`dict[str, Any] | None`:= Stores metadata attached to model traces. Its default value is `None`.

5. `rate_limiter`:`BaseRateLimiter | None`:= Stores an optional rate limiter applied before model execution. Its default value is `None`.

## Methods

1. `invoke`:= Returns the final message contained in the supplied input.

   When the input is a plain string, LangChain converts it into a `HumanMessage`, so the returned value may be a `HumanMessage` rather than an `AIMessage`.

   When multiple messages are supplied, all earlier messages are ignored.

   A `ValueError` is raised when no messages are supplied.

   **Syntax**

   ```python
   invoke(
       self, # Parrot fake chat-model instance
       input: LanguageModelInput, # String, prompt value, or message-like sequence
       config: RunnableConfig | None = None, # Runtime invocation configuration
       *,
       stop: list[str] | None = None, # Stop substrings
       **kwargs: Any # Additional invocation arguments
   ) -> BaseMessage
   ```

2. `ainvoke`:= Asynchronously returns the final message contained in the supplied input.

   **Syntax**

   ```python
   async ainvoke(
       self, # Parrot fake chat-model instance
       input: LanguageModelInput, # String, prompt value, or message-like sequence
       config: RunnableConfig | None = None, # Runtime invocation configuration
       *,
       stop: list[str] | None = None, # Stop substrings
       **kwargs: Any # Additional invocation arguments
   ) -> BaseMessage
   ```

3. `batch`:= Processes multiple inputs and returns the final message from each separate input.

   **Syntax**

   ```python
   batch(
       self, # Parrot fake chat-model instance
       inputs: list[LanguageModelInput], # Inputs to process
       config: RunnableConfig | list[RunnableConfig] | None = None, # Shared or per-input configuration
       *,
       return_exceptions: bool = False, # Return exceptions instead of raising them
       **kwargs: Any # Additional batch arguments
   ) -> list[BaseMessage]
   ```

4. `stream`:= Provides the standard inherited synchronous streaming interface.

   It does not divide the returned message using configurable character or word boundaries.

   **Syntax**

   ```python
   stream(
       self, # Parrot fake chat-model instance
       input: LanguageModelInput, # String, prompt value, or message-like sequence
       config: RunnableConfig | None = None, # Runtime invocation configuration
       *,
       stop: list[str] | None = None, # Stop substrings
       **kwargs: Any # Additional streaming arguments
   ) -> Iterator[BaseMessageChunk]
   ```


In [ ]:
from langchain_core.language_models.fake_chat_models import ParrotFakeChatModel # Import the parrot fake chat model
from langchain_core.messages import HumanMessage, SystemMessage # Import message classes

model = ParrotFakeChatModel( # Create the fake chat-model instance
    cache=False, # Disable response caching
    tags=["demo"], # Attach a tracing tag
    metadata={"purpose": "testing"} # Attach tracing metadata
) # Finish creating the model

string_response = model.invoke("Hello, LangChain!") # Convert the string to a HumanMessage and return it
print(string_response.type, string_response.content) # Display the returned message type and content

messages = [ # Define a sequence of input messages
    SystemMessage(content="Answer politely."), # Add the first system message
    HumanMessage(content="What is Python?") # Add the final human message
] # Finish defining the message list

last_message = model.invoke(messages) # Return only the final message from the sequence
print(last_message.type, last_message.content) # Display the final message type and content

async_response = await model.ainvoke("Async message") # Return the input asynchronously in Jupyter
print(async_response.content) # Display the asynchronous response content

batch_results = model.batch([ # Process multiple independent inputs
    "First input", # Provide the first plain-string input
    [HumanMessage(content="Second input")], # Provide the second message-list input
]) # Finish the batch invocation

print([message.content for message in batch_results]) # Display the final message from each input

for chunk in model.stream("Streamed message"): # Stream the returned message through the inherited interface
    print(chunk.content, end="") # Display the streamed chunk content

print() # Move to the next output line

try: # Test invocation with no messages
    model.invoke([]) # Supply an empty message sequence
except ValueError as error: # Catch the expected validation error
    print(error) # Display the error message

# Errors

The fake chat-model implementations may raise the following errors during model construction, invocation, streaming, or execution.

1. `FakeListChatModelError`:= Raised by `FakeListChatModel.stream()` or `FakeListChatModel.astream()` when streaming reaches the configured `error_on_chunk_number`.

2. `StopIteration`:= Raised when `GenericFakeChatModel` is invoked after its `messages` iterator has been exhausted.

3. `IndexError`:= May be raised when `FakeListChatModel` or `FakeMessagesListChatModel` is created with an empty response list and then invoked.

4. `ValueError`:= Raised by `ParrotFakeChatModel` when it receives no messages.

   It may also be raised while streaming from `GenericFakeChatModel` when the configured message contains an unsupported content form.

5. `pydantic.ValidationError`:= Raised during model construction when a required field is missing or a supplied value does not match the expected type.

6. `Callback, cache, rate-limiter, and Runnable exceptions`:= Exceptions produced by configured callbacks, caching systems, rate limiters, or Runnable configuration are allowed to propagate to the caller.


In [2]:
from pydantic import ValidationError # Import Pydantic's validation error
from langchain_core.language_models.fake_chat_models import ( # Import fake chat-model classes
    FakeListChatModel, # Import the sequential response model
    FakeListChatModelError, # Import the deliberate streaming error
    FakeMessagesListChatModel, # Import the complete-message response model
    GenericFakeChatModel, # Import the iterator-based response model
    ParrotFakeChatModel, # Import the model that returns the final input message
) # Finish importing the classes

error_model = FakeListChatModel( # Create a model that fails during streaming
    responses=["Hello"], # Define the response to stream
    error_on_chunk_number=2 # Raise an error before character index 2
) # Finish creating the model

try: # Handle the deliberate streaming failure
    for chunk in error_model.stream("Input"): # Stream the response character by character
        print(chunk.content, end="") # Display characters produced before the error
except FakeListChatModelError: # Catch the configured streaming error
    print("\nFakeListChatModelError raised.") # Display the error result

iterator_model = GenericFakeChatModel( # Create an iterator-based fake model
    messages=iter(["Only response"]) # Supply one predefined response
) # Finish creating the model

print(iterator_model.invoke("First input").content) # Consume and display the only response

try: # Test the exhausted iterator
    iterator_model.invoke("Second input") # Attempt to consume another response
except StopIteration: # Catch the exhausted-iterator error
    print("StopIteration raised.") # Display the error result

empty_model = FakeMessagesListChatModel( # Create a model with no responses
    responses=[] # Supply an empty response list
) # Finish creating the model

try: # Test invocation with an empty response list
    empty_model.invoke("Input") # Attempt to retrieve a configured response
except IndexError: # Catch the empty-list error
    print("IndexError raised.") # Display the error result

parrot_model = ParrotFakeChatModel() # Create the parrot fake chat model

try: # Test invocation without messages
    parrot_model.invoke([]) # Supply an empty message sequence
except ValueError as error: # Catch the missing-message error
    print(f"ValueError raised: {error}") # Display the error result

try: # Test invalid model construction
    FakeListChatModel() # Omit the required responses field
except ValidationError as error: # Catch the Pydantic validation error
    print("ValidationError raised.") # Display the error heading
    print(error.errors()[0]["msg"]) # Display the first validation message

He
FakeListChatModelError raised.
Only response
StopIteration raised.
IndexError raised.
ValueError raised: messages list cannot be empty.
ValidationError raised.
Field required
